# Kong Konnect: From Unprotected API to Secure, AI-Ready Platform

Welcome to this API Platform Engineering module! In this interactive notebook, we follow a **Construction** narrative. We will take a vulnerable, "Unprotected API" and systematically transform it into a Secure, AI-Ready Platform using Kong Konnect.

### Context:
- **Environment:** Kong Konnect (SaaS)
- **Configuration:** Reads variables dynamically from `.env` and `.deck.yaml`
- **Backend Service:** [Kong-Air-Flights-API](https://portal.kongair.dev/apis/flights-api/versions/74c7c2b2-fdaf-4ab3-9579-5f636eaa7370/operations/get-flights) (Host: `api.kongair.dev`)
- **CLI Tool:** `deck`

---
### Setup Requirements
Before we begin, ensure you have the `deck` CLI installed, a Konnect Serverless Gateway provisioned, and are authenticated with Konnect.

**Prerequisite: Serverless Gateway:**
You must already have a [Serverless Gateway deployed in Kong Konnect](https://developer.konghq.com/serverless-gateways/). You will need the Control Plane's Name and its generated Proxy URL for the configuration files below.

**Install decK:**
If you haven't installed `deck` yet, you can follow the instructions at [https://developer.konghq.com/deck/](https://developer.konghq.com/deck/) or run the following if you're on macOS:
```bash
brew tap kong/deck
brew install deck
```

**decK Configuration (`.deck.yaml`):**
Create a `.deck.yaml` file by copying the provided `.deck.yaml.template` file (see [configuration docs](https://developer.konghq.com/deck/configuration/#configuration-file)). 
```bash
cp .deck.yaml.template .deck.yaml
```
Open `.deck.yaml` and update the values with your credentials:
- `konnect-token`: Create a Personal Access Token (PAT) by following the instructions [here](https://developer.konghq.com/konnect-api/#personal-access-tokens).
- `konnect-control-plane-name`: This is the `Name` property displayed in the Gateway Manager overview for your specific Control Plane in the Konnect UI.
- `konnect-addr`: Keep this as `https://eu.api.konghq.com` unless your Control Plane is hosted in a different region.

**Environment Variables (`.env`):**
Create a `.env` file by copying the provided `.env.template` file:
```bash
cp .env.template .env
```
Update the `SERVERLESS_URL` in your new `.env` file to match the Proxy URL found in your Konnect control plane view.

In [ ]:
# Run this cell to install the required dependency for reading the .env file
%pip install python-dotenv -q

In [ ]:
# We will load our environment variables using python-dotenv.
# Ensure you have it installed: pip install python-dotenv

import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

SERVERLESS_URL = os.environ.get("SERVERLESS_URL")

# Set environment variables so %%bash commands can use them natively
os.environ["SERVERLESS_URL"] = SERVERLESS_URL or ""

print(f"Loaded Serverless URL: {SERVERLESS_URL}")
print("deck CLI will automatically pick up authentication details from .deck.yaml using the --config flag in our bash cells.")

### Clean Slate Initialization
Let's ensure we are starting with a completely empty Gateway configuration before proceeding. We will dynamically pipe an empty decK declaration to wipe any existing state.

In [ ]:
%%bash
cat <<EOF | deck gateway sync - --config .deck.yaml
_format_version: "3.0"
EOF

echo "Validating state..."
while ! cat <<EOF | deck gateway validate - --config .deck.yaml > /dev/null 2>&1
_format_version: "3.0"
EOF
do
    sleep 2
done
echo "Gateway has been successfully cleared!"

---
### Step 1: Create a Service and Route

**The Problem: "The Unprotected Service"**
Currently, our backend `Kong-Air-Flights-API` has no front door. If we expose it directly, anyone can access it, abuse it, or take it down. We need an API Gateway to act as a protective barrier.

**The Solution:**
We will create a **Service** (the logical representation of our backend) and a **Route** (the entry point on the gateway) in Kong Konnect using `deck`.

In [ ]:
%%bash
cat <<EOF > step1_service.yaml
_format_version: "3.0"
services:
  - name: Kong-Air-Flights-API
    host: api.kongair.dev
    port: 443
    protocol: https
    routes:
      - name: KongAir
        paths:
          - /kongair
EOF

deck gateway sync step1_service.yaml --config .deck.yaml

echo "Validating state..."
while ! deck gateway validate step1_service.yaml --config .deck.yaml > /dev/null 2>&1; do
    sleep 2
done
sleep 10 # Adding a buffer so changes fully propagate and take effect
echo "State validated!"

---
### Verification

Now that our gateway is routing traffic to the backend, let's test it.

**Expected Result:** A `200 OK` response because our API is completely open through the gateway.

In [ ]:
!curl -i {SERVERLESS_URL}/kongair/flights

---
### Step 2: Create a Consumer and Generate a Key

**The Problem: "Anonymous Access"**
Our API is currently completely public. Anyone who finds the URL can read all our data. We need to know *who* is calling our API. To do that, we first need to define a **Consumer** and give them a secret key.

**The Solution:**
We will use Python to automatically generate a secure API key, save it to our `.env` file, and then use `deck` to create the Consumer in Kong Konnect.

In [ ]:
import secrets
from dotenv import set_key

# Generate a random 32-character API key for our new consumer
CONSUMER_API_KEY = secrets.token_hex(16)
os.environ["CONSUMER_API_KEY"] = CONSUMER_API_KEY

# Update or append this new API key in the .env file so it persists for future runs
# Using set_key ensures we don't create duplicate entries if the cell is run multiple times
set_key(".env", "CONSUMER_API_KEY", CONSUMER_API_KEY)

print(f"Generated and saved new Consumer API Key: {CONSUMER_API_KEY}")

In [ ]:
%%bash
cat <<EOF > step2_consumer.yaml
_format_version: "3.0"
services:
  - name: Kong-Air-Flights-API
    host: api.kongair.dev
    port: 443
    protocol: https
    routes:
      - name: KongAir
        paths:
          - /kongair
consumers:
  - username: my_api_consumer
    keyauth_credentials:
      - key: $CONSUMER_API_KEY
EOF

deck gateway sync step2_consumer.yaml --config .deck.yaml

echo "Validating state..."
while ! deck gateway validate step2_consumer.yaml --config .deck.yaml > /dev/null 2>&1; do
    sleep 2
done
echo "State validated!"

---
### Step 3: Identity Guard (key-auth)

**The Problem: "Identity Enforcement"**
Even though we created a Consumer with an API Key, the API itself is still naked! If you test it now, anyone can still call it because we haven't instructed the gateway to check for keys.

**The Solution:**
We will attach the `key-auth` plugin to our service to enforce identity checks.

In [ ]:
%%bash
cat <<EOF > step3_keyauth.yaml
_format_version: "3.0"
services:
  - name: Kong-Air-Flights-API
    host: api.kongair.dev
    port: 443
    protocol: https
    plugins:
      - name: key-auth
        config:
          key_names:
            - apikey
    routes:
      - name: KongAir
        paths:
          - /kongair
consumers:
  - username: my_api_consumer
    keyauth_credentials:
      - key: $CONSUMER_API_KEY
EOF

deck gateway sync step3_keyauth.yaml --config .deck.yaml

echo "Validating state..."
while ! deck gateway validate step3_keyauth.yaml --config .deck.yaml > /dev/null 2>&1; do
    sleep 2
done
sleep 10 # Adding a buffer so changes fully propagate and take effect
echo "State validated!"

---
### Verification (Authentication)

Let's verify that anonymous access is now blocked, but authorized access works.

In [ ]:
print("Testing without API key (Should fail with 401 Unauthorized):")
!curl -s -o /dev/null -w "HTTP %{{http_code}}\n" {SERVERLESS_URL}/kongair/flights

print("\nTesting WITH API key (Should succeed with 200 OK):")
!curl -s -o /dev/null -w "HTTP %{{http_code}}\n" -H "apikey: {CONSUMER_API_KEY}" {SERVERLESS_URL}/kongair/flights

---
### Step 4: Traffic Control (rate-limiting)

**The Problem: "Service Melting"**
Even with authentication, a legitimate user could send millions of requests per second, taking down our backend.

**The Solution:**
We will apply a `rate-limiting` plugin to restrict traffic to 10 requests per minute.

In [ ]:
%%bash
cat <<EOF > step4_ratelimiting.yaml
_format_version: "3.0"
services:
  - name: Kong-Air-Flights-API
    host: api.kongair.dev
    port: 443
    protocol: https
    plugins:
      - name: key-auth
        config:
          key_names:
            - apikey
      - name: rate-limiting
        config:
          minute: 10
          policy: local
    routes:
      - name: KongAir
        paths:
          - /kongair
consumers:
  - username: my_api_consumer
    keyauth_credentials:
      - key: $CONSUMER_API_KEY
EOF

deck gateway sync step4_ratelimiting.yaml --config .deck.yaml

echo "Validating state..."
while ! deck gateway validate step4_ratelimiting.yaml --config .deck.yaml > /dev/null 2>&1; do
    sleep 2
done
sleep 10 # Adding a buffer so changes fully propagate and take effect
echo "State validated!"

---
### Verification (Rate Limiting)

We will trigger a burst of 12 requests using a shell loop. The first 10 should succeed, and the 11th and 12th should return a `429 Too Many Requests`.

In [ ]:
import subprocess

print("Sending 12 requests in a row to test rate limiting...")
for i in range(1, 13):
    cmd = f'curl -s -o /dev/null -w "%{{http_code}}" -H "apikey: {CONSUMER_API_KEY}" {SERVERLESS_URL}/kongair/flights'
    status = subprocess.check_output(cmd, shell=True).decode('utf-8').strip()
    print(f"Request {i}: HTTP {status}")

---
### Step 5: AI Transformation (ai-mcp-proxy)

**The Problem: "AI Readability"**
Modern consumers aren't just web apps; they are AI Agents. A standard REST API is hard for an AI to parse natively without clear function schemas.

**The Solution:**
We will transform our API into an AI-ready Platform by implementing the `ai-mcp-proxy` plugin. This exposes "Get Flights" and "Get Routes" as semantic tools for AI consumption.

Note: Rate Limiting Plugin is removed to allow MCP client to constantly call the MCP Server.

In [ ]:
%%bash
# We'll save this final YAML configuration to a file and validate it.
cat <<EOF > step5_ai_proxy.yaml
_format_version: "3.0"
services:
  - name: Kong-Air-Flights-API
    host: api.kongair.dev
    port: 443
    protocol: https
    plugins:
      - name: key-auth
        config:
          key_names:
            - apikey
    routes:
      - name: KongAir
        paths:
          - /kongair
        plugins:
          - name: ai-mcp-proxy
            config:
              consumer_identifier: username
              default_acl: null
              include_consumer_groups: false
              logging:
                log_audits: false
                log_payloads: true
                log_statistics: true
              max_request_body_size: 8192
              mode: conversion-listener
              server:
                forward_client_headers: true
                tag: null
                timeout: 10000
              tools:
                - acl: null
                  annotations:
                    destructive_hint: null
                    idempotent_hint: null
                    open_world_hint: null
                    read_only_hint: null
                    title: Get Flights
                  description: Get KongAir Flights
                  headers: null
                  host: null
                  method: GET
                  name: null
                  parameters: null
                  path: flights
                  query: null
                  request_body: null
                  responses: null
                  scheme: null
                - acl: null
                  annotations:
                    destructive_hint: null
                    idempotent_hint: null
                    open_world_hint: null
                    read_only_hint: null
                    title: Get Routes
                  description: Get KongAir Routes
                  headers: null
                  host: null
                  method: GET
                  name: null
                  parameters: null
                  path: routes
                  query: null
                  request_body: null
                  responses: null
                  scheme: null
consumers:
  - username: my_api_consumer
    keyauth_credentials:
      - key: $CONSUMER_API_KEY
EOF

deck gateway sync step5_ai_proxy.yaml --config .deck.yaml

echo "Validating state..."
while ! deck gateway validate step5_ai_proxy.yaml --config .deck.yaml > /dev/null 2>&1; do
    sleep 2
done
sleep 10 # Adding a buffer so changes fully propagate and take effect
echo "State validated!"

---
### Final Verification (AI-Ready API)

Because this endpoint now acts as an MCP server, we can verify it using the official MCP Inspector. You will need `npx` (Node.js) installed to run the inspector locally.

We will run `npx` to start the inspector and connect to our Konnect Proxy URL using the Server-Sent Events (SSE) transport type. 

**Important:** The Inspector UI does not automatically receive your API Key! Once the browser UI opens, you must manually add your `apikey` to the **Custom Headers** section under Authentication before connecting.

<img src="./resources/mcpInspectorHeaders.png" width="300" alt="MCP Inspector Authentication UI">

In [ ]:
import subprocess

cmd = f"npx -y @modelcontextprotocol/inspector --transport streamable-http --server-url '{SERVERLESS_URL}/kongair/mcp'"

print("Testing AI MCP Proxy Endpoint via MCP Inspector...")
print(f"Executing: {cmd}\n")
print(f"============================================================")
print(f"📋 COPY YOUR API KEY: {CONSUMER_API_KEY}")
print(f"============================================================\n")

print("Starting MCP inspector...")

# Start the MCP inspector as an interactive foreground process. 
# The user must click the 'Interrupt' (Stop) button in the notebook to terminate it.
try:
    process = subprocess.Popen(cmd, shell=True)
    
    print("\nThe inspector should automatically open a new browser tab.")
    print("Use the browser UI to interact with the 'Get Flights' and 'Get Routes' tools!")
    print(">>> NOTE: This cell will run continuously. Click 'Interrupt/Stop' in the notebook when you are finished to stop the server! <<<")
    
    # Wait indefinitely for the process to finish
    process.wait()
except (KeyboardInterrupt, SystemExit):
    print("\nInterrupt or termination signal received. Stopping the MCP Inspector...")
    process.terminate()
    process.wait()
    print("Inspector stopped.")